# Notebook 05 - Compare Base / Raw SFT / Adapted SFT

This notebook creates a qualitative comparison table for the three model conditions.

The important experimental discipline: use the same prompts and sampling settings for all three models.

In [ ]:
import sys

from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())

sys.path.append(str(ROOT / 'src'))

from kirundi_sft_starter.tinker_utils import generate_model_responses
from kirundi_sft_starter.utils import load_yaml, read_jsonl

project_config = load_yaml(ROOT / 'configs/project.yaml')

In [ ]:
print(project_config['models']['registry'])

## Generate comparable responses

This cell calls Tinker directly from the notebook and writes the standard response files under `results/responses/`. Run Notebooks 03 and 04 first so the two SFT checkpoint files exist.

In [ ]:
comparison_prompts = read_jsonl(ROOT / project_config['evaluation']['language_prompts_path'])

for model_key in project_config['models']['registry']:
    response_path = generate_model_responses(project_config, model_key, comparison_prompts)
    print(f'Wrote {response_path}')

In [ ]:
response_frames = []

for model_key, model_config in project_config['models']['registry'].items():
    response_path = ROOT / model_config['response_path']
    if response_path.exists():
        model_responses = pd.DataFrame(read_jsonl(response_path))
        model_responses['model'] = model_key
        response_frames.append(model_responses)

if response_frames:
    comparison_responses = pd.concat(response_frames, ignore_index=True)
    comparison_responses.head()
else:
    comparison_responses = pd.DataFrame()
    print('No response files yet. Run the response generation cell above.')

In [ ]:
if response_frames:
    qualitative_wide = comparison_responses.pivot_table(index=['prompt_id', 'prompt'], columns='model', values='response', aggfunc='first').reset_index()
    print(qualitative_wide.to_string(index=False))

In [ ]:
if response_frames:
    length_summary = comparison_responses.assign(response_chars=comparison_responses['response'].str.len()).groupby('model')['response_chars'].mean().reset_index()
    print(length_summary.to_string(index=False))

## Qualitative review questions

- Did the answer stay in Kirundi?
- Did it answer the question directly?
- Did it copy unwanted reasoning traces?
- Did Adaption appear to improve clarity or formatting?
- Are there examples that need native speaker review before any conclusion?